# Sesión 1: Entorno reproducible y manejo de bases de encuestas con pandas

**Seminario de Herramientas Estadísticas de Actualidad** · Estadística, Universidad Central del Ecuador

## Objetivos de la sesión

Al terminar este notebook podrás:

1. Comprobar que tu entorno de Python funciona y trabajar con rutas relativas al proyecto.
2. Explorar una base de encuesta con pandas y relacionar cada paso con lo que ya haces en R.
3. Limpiar y recodificar variables (códigos de "no sabe", etiquetas, categorías ordenadas).
4. Calcular medias **con y sin factor de expansión** y ver por qué importa la diferencia.
5. Guardar la base limpia en `data/processed/`.

La base es **sintética**: no representa a ninguna población real. Sirve para que todos obtengan los mismos resultados sin descargar nada. En el proyecto integrador usarás una encuesta oficial real.

Tabla de apoyo: `docs/equivalencias_R_Python.md`.

## 0. Celda de arranque (ejecútala primero, cada vez)

En Colab, la sesión se borra al cerrarla. Esta celda trae de nuevo los archivos de la plantilla (carpeta `src/`) y deja las rutas listas. En una instalación local no hace nada.

In [ ]:
# CELDA DE ARRANQUE: ejecútala primero en cada sesión de Colab.
import os
import subprocess
import sys

REPO_URL = "https://github.com/USUARIO/seminario-python-encuestas.git"  # el docente indica el enlace real
CARPETA = "/content/seminario"

if "google.colab" in sys.modules:
    if "USUARIO" in REPO_URL:
        raise RuntimeError("Falta el enlace de la plantilla: el docente lo indicará.")
    if not os.path.exists(CARPETA):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CARPETA], check=True)
    os.chdir(os.path.join(CARPETA, "notebooks"))
    print("Plantilla lista en Colab:", os.getcwd())
else:
    print("Instalación local: no hace falta arrancar nada.")

## 1. Verificación del entorno

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)

Para poder importar los módulos de `src/`, necesitamos que Python encuentre la **raíz del proyecto**. Este notebook vive en `notebooks/`, así que la raíz es la carpeta de arriba. Usamos rutas relativas con `pathlib`: así el código funciona igual en tu computador y en el de cualquier compañero.

In [ ]:
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RUTA_PROCESADOS = RAIZ / "data" / "processed"
RUTA_SALIDAS = RAIZ / "outputs"

print("Raíz del proyecto:", RAIZ.name)
print("Existe data/processed:", RUTA_PROCESADOS.exists())

> **Comparación con R.** En R usarías `here::here()` o un proyecto de RStudio para fijar la raíz. En Python, `pathlib.Path` cumple ese papel. El principio es el mismo: **nunca escribir rutas absolutas** como `C:/Users/...`.

## 2. Cargar los datos

En la práctica leerías la encuesta con `pd.read_csv(...)` o `pd.read_spss(...)`. Aquí generamos la base sintética.

In [ ]:
from src.datos_sinteticos import (
    generar_encuesta, PROVINCIAS, AREAS, SEXOS, NIVELES, CODIGO_NO_SABE,
)

df = generar_encuesta(n=2000, semilla=2026)
df.head()

## 3. Primer vistazo a la base

| R | Python |
| --- | --- |
| `dim(df)` | `df.shape` |
| `str(df)` | `df.info()` |
| `summary(df)` | `df.describe()` |

In [ ]:
print("Filas y columnas:", df.shape)
df.info()

In [ ]:
df.describe()

Fíjate en dos cosas que en una encuesta real también aparecerían:

- `ingreso` tiene **valores faltantes** (`NaN`).
- `nivel_instr` incluye el código **99**, que significa "no sabe / no responde". Si no lo limpiamos, pandas lo tratará como un nivel de instrucción más y arruinará cualquier resumen.

In [ ]:
print("Faltantes por columna:")
print(df.isna().sum())
print()
print("Frecuencias de nivel_instr:")
print(df["nivel_instr"].value_counts().sort_index())

## 4. Seleccionar, filtrar y crear columnas

| R (dplyr) | Python (pandas) |
| --- | --- |
| `select(df, edad, ingreso)` | `df[["edad", "ingreso"]]` |
| `filter(df, edad >= 18)` | `df[df["edad"] >= 18]` o `df.query("edad >= 18")` |
| `mutate(df, ingreso_anual = ingreso * 12)` | `df.assign(ingreso_anual=df["ingreso"] * 12)` |
| `arrange(df, desc(ingreso))` | `df.sort_values("ingreso", ascending=False)` |

In [ ]:
adultos = df.query("edad >= 18")
print("Personas de 18 años o más:", len(adultos))

adultos[["id_persona", "edad", "ingreso"]].sort_values("ingreso", ascending=False).head()

Un detalle que sorprende al venir de R: en pandas se puede **encadenar métodos** con `.` y paréntesis, que cumple el papel del pipe `%>%`. Cada línea transforma el resultado de la anterior.

In [ ]:
(
    df.query("edad >= 18")
      .assign(ingreso_anual=lambda d: d["ingreso"] * 12)
      .sort_values("ingreso_anual", ascending=False)
      [["id_persona", "ingreso", "ingreso_anual"]]
      .head()
)

## 5. Limpieza y recodificación

Vamos a: (1) convertir el código 99 en valor faltante, (2) poner etiquetas legibles y (3) declarar `nivel_instr` como **categoría ordenada**, de modo que los resultados salgan en el orden lógico y no en el alfabético.

| R | Python |
| --- | --- |
| `na_if(x, 99)` | `x.replace(99, np.nan)` |
| `recode()` / etiquetas | `x.map(diccionario)` |
| `factor(x, levels = ..., ordered = TRUE)` | `pd.Categorical(x, categories=..., ordered=True)` |

In [ ]:
limpia = df.copy()   # nunca modificamos el original

# 1. El código 99 es "no sabe": lo pasamos a faltante
limpia["nivel_instr"] = limpia["nivel_instr"].replace(CODIGO_NO_SABE, np.nan)

# 2. Etiquetas legibles a partir de los diccionarios
limpia["provincia"] = limpia["prov"].map(PROVINCIAS)
limpia["area_etq"] = limpia["area"].map(AREAS)
limpia["sexo_etq"] = limpia["sexo"].map(SEXOS)

# 3. Categoría ordenada para el nivel de instrucción
orden = list(NIVELES.values())
limpia["nivel_etq"] = pd.Categorical(
    limpia["nivel_instr"].map(NIVELES),
    categories=orden,
    ordered=True,
)

limpia[["id_persona", "provincia", "area_etq", "sexo_etq", "nivel_etq", "ingreso", "fexp"]].head()

In [ ]:
# Comprobación: ya no hay códigos 99 y los faltantes quedaron como NaN
print("¿Quedan 99?:", (limpia["nivel_instr"] == CODIGO_NO_SABE).any())
print("Faltantes en nivel_etq:", limpia["nivel_etq"].isna().sum())
limpia["nivel_etq"].value_counts().sort_index()

## 6. Resumir por grupos: sin pesos y con pesos

En una encuesta cada fila **representa a muchas personas**, y el factor de expansión (`fexp`) dice cuántas. Si calculas una media sin ponderar, estás describiendo solo a la muestra, no a la población.

| R | Python |
| --- | --- |
| `group_by(g) %>% summarise(m = mean(x))` | `df.groupby("g").agg(m=("x", "mean"))` |
| `weighted.mean(x, w)` | `np.average(x, weights=w)` |

Primero, la media **sin ponderar** del ingreso por provincia:

In [ ]:
sin_pesos = (
    limpia.dropna(subset=["ingreso"])
          .groupby("provincia")
          .agg(media_sin_pesos=("ingreso", "mean"), n_muestra=("ingreso", "size"))
)
sin_pesos

Ahora la media **ponderada**. La calculamos como la suma de `ingreso × fexp` dividida para la suma de `fexp`, siempre sobre las personas que sí tienen ingreso (no se pueden ponderar los faltantes).

In [ ]:
con_ingreso = limpia.dropna(subset=["ingreso"]).assign(
    ingreso_x_peso=lambda d: d["ingreso"] * d["fexp"]
)

ponderada = (
    con_ingreso.groupby("provincia")
               .agg(suma_ingreso_peso=("ingreso_x_peso", "sum"),
                    suma_pesos=("fexp", "sum"))
)
ponderada["media_ponderada"] = ponderada["suma_ingreso_peso"] / ponderada["suma_pesos"]
ponderada[["media_ponderada"]]

In [ ]:
comparacion = sin_pesos.join(ponderada[["media_ponderada"]])
comparacion["diferencia"] = comparacion["media_ponderada"] - comparacion["media_sin_pesos"]
comparacion.round(2).sort_values("media_ponderada", ascending=False)

La diferencia existe porque el diseño **no es autoponderado**: la zona rural recibe pesos mayores y tiene ingresos más bajos. Ignorar `fexp` sesga el resultado hacia lo que pasa en la zona urbana.

> **Importante.** Esto da la *estimación puntual* correcta. Los **errores estándar** de una encuesta compleja requieren el diseño muestral (estratos, conglomerados); eso ya lo viste en Muestreo I y II y no se repite aquí. Para estimarlos en Python se usan librerías como `samplics`, que veremos si el proyecto lo necesita.

In [ ]:
# Una verificación rápida con la función de numpy, para un solo caso
guayas = con_ingreso[con_ingreso["provincia"] == "Guayas"]
print("Media ponderada de Guayas con np.average:",
      round(np.average(guayas["ingreso"], weights=guayas["fexp"]), 2))
print("Media ponderada de Guayas con la tabla:  ",
      round(ponderada.loc["Guayas", "media_ponderada"], 2))

## 7. Un primer gráfico

La sesión 3 está dedicada a visualización; aquí solo un gráfico simple para cerrar el ciclo: comparar la media ponderada con la no ponderada.

In [ ]:
orden_prov = comparacion.sort_values("media_ponderada").index
x = np.arange(len(orden_prov))
ancho = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(x - ancho / 2, comparacion.loc[orden_prov, "media_sin_pesos"], ancho,
        label="Sin ponderar")
ax.barh(x + ancho / 2, comparacion.loc[orden_prov, "media_ponderada"], ancho,
        label="Ponderada")
ax.set_yticks(x)
ax.set_yticklabels(orden_prov)
ax.set_xlabel("Ingreso mensual medio (USD)")
ax.set_title("Ingreso medio por provincia (datos sintéticos)")
ax.legend()
fig.tight_layout()

RUTA_SALIDAS.mkdir(exist_ok=True)
fig.savefig(RUTA_SALIDAS / "sesion01_ingreso_provincia.png", dpi=150)
plt.show()

## 8. Guardar la base limpia

La base original no se toca. Lo que limpiaste se guarda en `data/processed/`, de modo que los notebooks siguientes parten de esa base ya limpia.

In [ ]:
RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
destino = RUTA_PROCESADOS / "encuesta_sintetica_limpia.csv"
limpia.to_csv(destino, index=False)
print("Guardado:", destino.relative_to(RAIZ), "·", len(limpia), "filas")

## Ejercicios

Resuélvelos en celdas nuevas debajo de cada enunciado. Guarda este notebook con tus respuestas.

**Ejercicio 1.** Calcula la **mediana** del ingreso por área (urbana y rural), sin ponderar. ¿Qué área tiene la mediana más alta?

**Ejercicio 2.** Calcula el porcentaje **ponderado** de personas con nivel de instrucción "Superior" por provincia, entre quienes tienen nivel de instrucción declarado. *Pista: la suma de `fexp` de quienes tienen "Superior" dividida para la suma de `fexp` de quienes tienen nivel declarado.*

**Ejercicio 3.** Crea una columna `grupo_edad` con `pd.cut` (por ejemplo 15-24, 25-44, 45-64 y 65 o más) y calcula la media ponderada del ingreso por grupo de edad.

**Ejercicio 4 (reto).** Escribe una función `media_ponderada_por(df, grupo, variable, peso)` que devuelva la media ponderada de `variable` por cada valor de `grupo`, ignorando los faltantes. Pruébala con las provincias y comprueba que coincide con la tabla de la sección 6.

In [ ]:
# Ejercicio 1

In [ ]:
# Ejercicio 2

In [ ]:
# Ejercicio 3

In [ ]:
# Ejercicio 4

## Para la próxima sesión

- Elige la **encuesta de tu proyecto integrador** (ENEMDU, ENSANUT u otra aprobada) y descárgala del INEC a `data/raw/`.
- Revisa el diccionario de variables y localiza: el **factor de expansión**, los **códigos geográficos** (provincia, cantón, parroquia) y el indicador que quieres mapear.
- Verifica que el código geográfico de tu base coincida con el de la cartografía oficial; lo usaremos en la semana 4 para unir ambas.